# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chapcoda/flyrank-ML-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship-starter


In [2]:
!pip install -q duckdb huggingface_hub

from huggingface_hub import login
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")  # stored in Colab Secrets, never in the notebook
login(token=HF_TOKEN)


In [3]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

DATASET = "hf://datasets/FlyRank/internship-warehouse"

# Sanity check — should return 104
q = f"SELECT COUNT(*) AS row_count FROM read_parquet('{DATASET}/dim_clients.parquet')"
con.execute(q).fetchdf()

,row_count
0,104


## 1. Unit of analysis + time window


**Unit of analysis:** one row = one **content page** (identified by `content_hash_id`, linked to one `client_hash_id` — the website that owns the page).

**Where that grain comes from:** the source table `fact_content_daily_performance` is at the grain of *page × client × day* — one entry per page per website per day. My model needs to score *pages*, not page-days, so I aggregate this daily table into one row per page by computing features across a defined window.

**Time windows (fixed up-front, non-overlapping to prevent leakage):**

- **Feature window:** 2026-02-01 → 2026-04-30. Three months of history used to compute the model's inputs (impressions, clicks, engagement, etc.).
- **Target window:** 2026-05-01 → 2026-05-31. The 31 days whose outcome the model is trying to predict — did this page decline in that window or not?

The feature window ends April 30, the target window starts May 1. No overlap. This is the honest future-window label the lane guide pointed toward, not a same-window bucket like the starter's `is_declining_label`.

**Development slice used to build this contract:** the March 2026 partition of `fact_content_daily_performance`. Using one month during development keeps queries fast and avoids Hugging Face rate limits. The full feature window will be pulled only when features are computed for real.

**What March 2026 looks like on its own:** ~9.8M daily rows, ~331k unique pages, 55 unique clients (of 104 total in the warehouse), covering exactly March 1–31.

In [4]:
q_march = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM read_parquet('{DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet')
"""

con.execute(q_march).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_triples
    9841378         9841378

Grain claim holds? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field bucketing — feature / label / context / excluded

I've sorted every column from `dim_content` and `fact_content_daily_performance` into one of four buckets. Features are what the model uses to make its prediction. The label is what the model is trying to guess. Context is what I look at during exploration or use for validation grouping — but never let the model see. Excluded fields are deliberately kept out, with a reason.

#### Features (aggregated over the feature window: 2026-02-01 to 2026-04-30)

Page-shape features from `dim_content` (mostly static, one row per page):

| Column | Type | Why it's a feature |
|---|---|---|
| `keyword_char_count`, `keyword_token_count` | numeric | Length of the target keyword — longer keywords tend to be lower-competition. |
| `url_char_count` | numeric | URL length — a rough proxy for site depth. |
| `content_type` | categorical | What kind of page (article, comparison, etc.). |
| `search_volume` | numeric | Demand for the keyword. |
| `competition`, `competition_level` | numeric + categorical | How hard the keyword is to rank for. |
| `cpc` | numeric | Commercial value of the keyword. |
| `main_intent` | categorical | What searchers want (informational, transactional, etc.). |
| `backlinks` | numeric | Link authority. |
| `category_count` | numeric | Category grouping signal. |
| `char_count`, `word_count` | numeric | Page length. |
| `content_age_days` | derived | Computed from `content_created_date` and the end of the feature window. |
| `days_since_last_update` | derived | Computed from `content_updated_date` and the end of the feature window. |
| `days_since_last_optimized` | derived | Computed from `last_optimized_date` and the end of the feature window. |

Performance features from `fact_content_daily_performance` (aggregated across the 90-day feature window):

| Column | Aggregation | Why it's a feature |
|---|---|---|
| `gsc_impressions` | sum over window | Total exposure in search results. |
| `gsc_clicks` | sum over window | Total click volume from search. |
| `gsc_avg_position` | mean over window (weighted by impressions) | Average ranking during the window. |
| `ga4_sessions` | sum over window | Total on-site visits (where GA4 was available). |
| `ga4_engaged_sessions` | sum over window | Meaningful visits. |
| `ga4_total_engagement_sec` | sum over window | Time spent on the page. |
| `sessions_organic` | sum over window | Organic search traffic — the main channel for content SEO. |
| Derived: `ctr` | gsc_clicks / gsc_impressions | Click-through rate, standard SEO measure. |
| Derived: `engagement_rate` | ga4_engaged_sessions / ga4_sessions | Engagement quality. |

#### Label (computed over the target window: 2026-05-01 to 2026-05-31)

| Column | Why it's the label |
|---|---|
| `is_declining_label` (derived) | Binary flag: **1 if the page's total `gsc_impressions` in the target window is at least 25% lower than the trailing 30 days of the feature window; else 0.** This is a *future-window observed outcome* — the honest label the lane guide pointed toward, not the starter's proxy `trend_direction == "down"`. Threshold (25%) and window lengths chosen up-front; policy documented. |

#### Context (used for grouping / validation / EDA, never as model features)

| Column | Why it's context |
|---|---|
| `client_hash_id` | Grouping only. The model must generalize across clients, so client identity is used for train/test splits (client-holdout), never as a feature. Feeding it in would let the model memorize clients. |
| `content_hash_id` | Row identifier. Grouping only. |
| `keyword_hash_id`, `url_hash_id` | Group-level dedup and case analysis. Never used as features — they're identifiers, not signals. |
| `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` | Availability flags. Used to correctly interpret missing values (null ≠ zero), not as features. |
| `report_date` | Used to define feature/target windows. Never a feature itself. |

#### Excluded (deliberately not used — with a reason)

| Column | Reason for exclusion |
|---|---|
| `content_created_date`, `content_updated_date`, `keyword_created_date`, `last_optimized_date`, `optimization_eligible_date` | Raw dates would leak absolute time. I derive age/recency features from them (see features table above) instead of feeding the dates in directly. |
| `is_published`, `is_deleted` | Filter fields, not features. I keep only rows where `is_published = TRUE` and `is_deleted = FALSE`. |
| `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | Too sparse to be useful features. The lane guide flagged only ~30k rows out of 79M have any AI sessions at all. Including these adds noise. Kept for future EDA / freestyle direction only. |
| `sessions_paid`, `sessions_social`, `sessions_referral`, `sessions_direct` | Non-organic traffic sources. My target is *organic* search performance (that's what "declining" means in this project). Paid/social/direct sessions are a different signal and would confuse the model about what it's predicting. |
| `gsc_sum_position` | Redundant with `gsc_avg_position` (the two encode the same information; the average is the standard one to use). |
| `provider_used`, `model_used` | Metadata about content-generation tooling. Would risk letting the model discriminate on how content was produced rather than on what the content *is*. Excluded for fairness. |
| `ga4_pageviews`, `ga4_users` | Highly correlated with `ga4_sessions`. Including all three adds no information and would confuse feature-importance analysis. |
| `scroll_events` | Very sparse in the density table (only ~600k rows have scroll data across 79M total). Not enough signal to justify inclusion. |
| `month` | Partitioning artifact. Not real data. |

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q_grain_content = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages
FROM read_parquet('{DATASET}/dim_content.parquet')
"""
result = con.execute(q_grain_content).fetchdf()
print(result.to_string(index=False))
print()
print("Grain claim holds?", result['total_rows'].iloc[0] == result['unique_pages'].iloc[0])

 total_rows  unique_pages
     519606        519606

Grain claim holds? True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim in Sections 1 and 2 gets a verification query below. A contract claim without a proof is a guess.

**V1 — grain of fact table.** Claim: one row = one (page × client × day). Query counts total rows vs. unique (content_hash_id, client_hash_id, report_date) triples.

**Result:** 9,841,378 in both. **Grain claim holds.** No duplication. Safe to sum/average across the window without deduplication.

In [8]:
q_grain_fact = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (content_hash_id, client_hash_id, report_date)) AS unique_triples
FROM read_parquet('{DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet')
"""
result = con.execute(q_grain_fact).fetchdf()
print(result.to_string(index=False))
print()
print("Grain claim holds?", result['total_rows'].iloc[0] == result['unique_triples'].iloc[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_triples
    9841378         9841378

Grain claim holds? True


**V2 — grain of `dim_content`.** Claim: one row = one unique page. Query counts total rows vs. unique `content_hash_id`.

**Result:** 519,606 in both. **Grain claim holds.** Every row is a unique page. Safe to join against the fact table without inflating row counts.

In [ ]:
q_grain_content = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages
FROM read_parquet('{DATASET}/dim_content.parquet')
"""
result = con.execute(q_grain_content).fetchdf()
print(result.to_string(index=False))
print()
print("Grain claim holds?", result['total_rows'].iloc[0] == result['unique_pages'].iloc[0])

**V3 — GA4 availability honesty.** Claim: `ga4_data_available = FALSE` should mean `NULL` in the GA4 columns (honest missing), not `0` (dishonest zero). Query groups by the flag and counts nulls vs. zeros in `ga4_sessions` and `ga4_pageviews`.

**Result — three populations:**
- **`FALSE`** (6.41M rows, 65% of March): GA4 columns store `0`, not `NULL`. **This is dishonest zeros** — treating "we couldn't measure" the same as "no visitors."
- **`NULL` flag** (3.02M rows, 31%): the flag itself is missing. Here GA4 columns are `NULL` — honest missing.
- **`TRUE`** (0.41M rows, 4%): real GA4 data.

**Consequence:** only ~4% of March rows have honestly measured GA4 metrics. When aggregating GA4 features, must filter to `ga4_data_available = TRUE` (or mask GA4 columns to `NULL` when the flag is anything else). Primary features should come from GSC; GA4 features are secondary.

In [9]:
q_ga4_null_check = f"""
SELECT
    ga4_data_available,
    COUNT(*) AS row_count,
    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS null_sessions,
    SUM(CASE WHEN ga4_sessions = 0 THEN 1 ELSE 0 END) AS zero_sessions,
    SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS null_pageviews,
    SUM(CASE WHEN ga4_pageviews = 0 THEN 1 ELSE 0 END) AS zero_pageviews
FROM read_parquet('{DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY ga4_data_available
ORDER BY ga4_data_available
"""
con.execute(q_ga4_null_check).fetchdf()

,ga4_data_available,row_count,null_sessions,zero_sessions,null_pageviews,zero_pageviews
0,False,6408671,0.0,6408671.0,0.0,6408671.0
1,True,413966,0.0,3631.0,0.0,649.0
2,<NA>,3018741,3018741.0,0.0,3018741.0,0.0


**V4 — GSC availability honesty.** Same check on the GSC side. Query groups by `gsc_data_available` and counts nulls vs. zeros in `gsc_impressions` and `gsc_clicks`.

**Result — two populations (no NULL-flag group):**
- **`FALSE`** (6.23M rows, 63% of March): GSC columns store `0`, not `NULL`. **Same dishonest-zeros trap as GA4.**
- **`TRUE`** (3.61M rows, 37% of March): real GSC data. All measured rows have at least one impression. Zero clicks in ~88% of measured rows is legitimate (pages showed up in search but weren't clicked — normal for long-tail SEO).

**Consequence:** GSC coverage (~37%) is ~9× better than GA4 coverage (~4%). This confirms the Section 2 plan: primary features come from GSC (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`). GA4 features stay secondary. Same filtering rule applies: mask to `NULL` (or filter to `TRUE`) when `gsc_data_available = FALSE`.

In [ ]:
q_gsc_null_check = f"""
SELECT
    gsc_data_available,
    COUNT(*) AS row_count,
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
    SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions,
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
    SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) AS zero_clicks
FROM read_parquet('{DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY gsc_data_available
ORDER BY gsc_data_available
"""
con.execute(q_gsc_null_check).fetchdf()

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The point of this section is honesty about what my model can and can't be asked to do. Every limit below is structural — no amount of modeling or feature engineering will paper over these gaps. Naming them now prevents me from writing conclusions later that the data can't support.

**1. Coverage limits — the population my data actually sees.**

- **Unbalanced client history.** Only 55 of the 104 clients in `dim_clients` appear in March 2026 (Section 1 query). The rest either onboarded later or offboarded earlier. This means my dev slice is not representative of "all clients" — it's representative of "clients active in March 2026." Any client-level story I tell has to be qualified accordingly.
- **Deep time-series analysis is limited to a subset of clients.** The lane guide notes only 9 of 70 clients have 12+ months of history. My feature window is only 3 months, so I sidestep this — but I cannot do full-year seasonality on most clients.
- **The population is content pages, not users.** I can rank pages by decline risk. I cannot infer anything about *who* was reading them or what they wanted differently.

**2. Availability limits — signals I can only measure sometimes.**

- **GA4 coverage is ~4% of rows.** V3 showed that only 413,966 of 9.8M March rows have `ga4_data_available = TRUE`. Everything I say about on-site engagement (sessions, engaged sessions, time on page) applies only to that ~4% subset. On the other 96% of pages, I have no view of user behavior at all — and if I'm not careful, treating the dishonest zeros as real would fabricate a signal that isn't there.
- **GSC coverage is ~37% of rows.** Better than GA4 but still not universal. My primary features (impressions, clicks, position) live in this ~37% subset. Pages outside this subset either can't be scored, or need imputation policies I'd have to defend separately.
- **AI-referral data is effectively unusable at this scale.** The lane guide flagged ~30k rows with AI sessions out of 79M total across all months (Section 2 excluded these). In the March slice, they'd be an even smaller fraction. Any story about "AI is driving traffic to this page" is off-limits with this data volume — I don't have enough non-zero examples to make it a feature.
- **Scroll events are similarly too sparse to trust.** ~600k rows out of 79M across all months. Excluded from features for the same reason.
- **Content-generation metadata (`provider_used`, `model_used`) is unusable for fairness reasons.** Even where it exists, I choose not to model on it — I don't want the model discriminating on how content was produced.

**3. Structural limits — things the data design fundamentally cannot answer.**

- **No causal claims.** This is observational data — I watch what happened, I don't run experiments. So I can rank pages by *decline risk* (correlation, association), but I cannot claim that *editing* a flagged page will *cause* it to recover. Proving that would need an A/B test or a matched intervention study, which is out of scope for this data.
- **No Google-algorithm claims.** I can describe patterns in how pages perform in search, but I cannot reverse-engineer why Google's ranking model behaved a certain way. Any claim of the form "Google is rewarding X" would be beyond what the data supports.
- **No individual-user behavior claims.** GA4 aggregates. I can see engaged sessions, but I can't see who they are, what they clicked next, or what they returned for.
- **No claim about pages I've never seen.** My model will be trained on 55 clients' active pages in a particular window. It will make honest predictions on similar pages from similar clients. On a brand-new client with different content types, my numbers should be treated as directional at best.

**4. Freshness cutoff.**

The daily fact table ends 2026-06-30 (the release cutoff). Anything I want to say about "the last week" or "recent trends" is off-limits — I have no data for it. My target window (2026-05-01 → 2026-05-31) sits comfortably inside the release, which is why I chose it.

**5. Non-overlapping windows are enforced by construction.**

The feature window (2026-02-01 → 2026-04-30) and target window (2026-05-01 → 2026-05-31) don't overlap by design. This eliminates same-window leakage (the trap I hit in notebook 02 with `trend_pct`). But I still need feature-engineering discipline — no feature derived from data inside the target window should ever land in the feature set.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.